# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. output: 20 10 — s["b"] берёт значение по метке индекса 'b', а s.iloc[0] — по позиции, первый элемент это 10.
2. output: Алматы NaN, Астана NaN, Шымкент 3.0 — pandas складывает по совпадающим меткам индекса; Шымкент есть в обеих (2+1=3), у остальных пары нет, поэтому NaN (и тип становится float).
3. output: (2, 2) — условие x > 1 выполняется для 2 строк (x=2 и x=3), а столбцов остаётся 2 (x и y).
4. output: Series DataFrame — одна квадратная скобка с именем столбца даёт Series, а список столбцов [["x"]] даёт DataFrame.
5. output: [1, 2] 5 — после фильтра остаются строки с исходными индексами 1 и 2 (индекс не сбрасывается); iloc[0, 1] — первая строка фильтра (x=2, y=5), второй столбец y, то есть 5.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [110]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [3]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [60]:
# 1
print("Размер таблицы (строки, столбцы):", df.shape)
print("\n--- Информация о типах и пропусках ---\n")
df.info()

missing_delivery = df["delivery_days"].isna().sum()
print(f"\nКоличество пропусков в delivery_days: {missing_delivery}")

print("\n--- Количество уникальных значений ---\n")
print(df[["city", "category", "status", "customer_id"]].nunique())

print("\n--- Детальный разбор значений по категориям, статусам и городам ---\n")
for col in ["city", "category", "status"]:
    print(f"\n[{col}]")
    print(df[col].value_counts(dropna=False))

Размер таблицы (строки, столбцы): (5463, 8)

--- Информация о типах и пропусках ---

<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 341.6 KB

Количество пропусков в delivery_days: 160

--- Количество уникальных значений ---

city             14
category          5
status            2
customer_id    2003
dtype: int64

--- Детальный разбор значений по категориям, статусам и городам ---


[city]
city
Алматы       1927
Астана       1226
Шымкент       711
Карага

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [ ]:
# 2
print("Среднее значение для оплаченных заказов без TEST:")
print(df[(df["status"] == "оплачен") & (df["customer_id"] != "TEST")].mean(numeric_only=True)["amount"])

print("\nМедианное значение для оплаченных заказов без TEST:")
print(int(df[(df["status"] == "оплачен") & (df["customer_id"] != "TEST")].median(numeric_only=True)["amount"]))

print("\n25-й перцентиль для оплаченных заказов без TEST:")
print(df[(df["status"] == "оплачен") & (df["customer_id"] != "TEST")].quantile(0.25, numeric_only=True)["amount"])

print("\n75-й перцентиль для оплаченных заказов без TEST:")
print(df[(df["status"] == "оплачен") & (df["customer_id"] != "TEST")].quantile(0.75, numeric_only=True)["amount"])

print("\nТипичный чек: центральное значение (медиана) составляет 10300 тг,\n а основной диапазон обычных покупок (межквартильный размах от 25-го до 75-го перцентиля) лежит в пределах от 5880.0 тг до 18690.0 тг\n — в этот коридор попадают 50% всех стандартных оплаченных заказов.")
print("\nЧестнее опираться на медиану (10300 тг), а не на среднее арифметическое (14672.391348704346 тг)." \
"\n Среднее сильно завышено относительно медианы из-за перекоса распределения (асимметрии): редкие, но очень дорогие заказы (например, техника и электроника) «тянут» среднее значение вверх.\n" \
" Медиана показывает реальную середину (половина клиентов покупает дешевле этой суммы, половина — дороже) и не искажается единичными крупными чеками.")


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [109]:
# 3
rejected = df["status"] == "отменён"
ALA = ['Алматы', 'алматы', 'АЛМАТЫ', ' Алматы', 'Almaty']

all_ = rejected[not_test].mean()
shym = rejected[not_test & df["city"].isin(SH)].mean()
ala = rejected[not_test & df["city"].isin(ALA)].mean()

print(f"Доля отмен по всем заказам: {all_ * 100:.2f}%")
print(f"Доля отмен в Шымкенте: {shym * 100:.2f}%")
print(f"Доля отмен в Алматы: {ala * 100:.2f}%")
print(f"\nШымкент vs средний показатель: в {shym / all_:.1f} раза выше")
print(f"Шымкент vs Алматы: в {shym / ala:.1f} раза выше")

Доля отмен по всем заказам: 10.24%
Доля отмен в Шымкенте: 20.40%
Доля отмен в Алматы: 7.87%

Шымкент vs средний показатель: в 2.0 раза выше
Шымкент vs Алматы: в 2.6 раза выше


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [ ]:
# 4

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [ ]:
# 5

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [ ]:
# 6

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8